# SC08 Predictive Maintenance System - Data Inspection & Pipeline

This notebook inspects the raw AI4I 2020 Predictive Maintenance dataset, performs quality checks, validates domain constraints, executes stratified splitting, and validates generated testing scenarios.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_FILE = PROJECT_ROOT / "data" / "raw" / "ai4i2020.csv"
print(f"Loading raw data from: {RAW_FILE}")

df = pd.read_csv(RAW_FILE)

print("SC08 PREDICTIVE MAINTENANCE RAW DATA INSPECTION")
print("-" * 50)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print("\nMachine failure class distribution:")
print(df["Machine failure"].value_counts().sort_index())
print("\nFirst five rows:")
display(df.head())

Loading raw data from: C:\Users\DEEPAK\OneDrive\New folder\SC08-Predictive-Maintenance-System\data\raw\ai4i2020.csv
SC08 PREDICTIVE MAINTENANCE RAW DATA INSPECTION
--------------------------------------------------
Dataset shape: 10000 rows, 14 columns

Column names:
['UDI', 'Product ID', 'Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Missing values per column:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

Duplicate rows: 0

Machine failure class distribution:
Machine failure
0    9661
1     339

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Step 2B: Clean the dataset without changing the original raw file

In [2]:
# Define features and working copy
feature_columns = [
    "Air temperature [K]",
    "Process temperature [K]",
    "Rotational speed [rpm]",
    "Torque [Nm]",
    "Tool wear [min]"
]

clean_df = df.copy()

# Record condition before cleaning
missing_before = clean_df.isna().sum()
duplicates_before = int(clean_df.duplicated().sum())

# Median imputation for numeric features (safe practice)
clean_df[feature_columns] = clean_df[feature_columns].fillna(clean_df[feature_columns].median())

# Deduplicate
clean_df = clean_df.drop_duplicates().reset_index(drop=True)

print("DATA CLEANING SUMMARY")
print("-" * 45)
print(f"Rows before cleaning: {len(df)}")
print(f"Rows after cleaning:  {len(clean_df)}")
print(f"Duplicate rows removed: {duplicates_before}")
print("\nMissing values after cleaning:")
print(clean_df.isna().sum())

print("\nFirst five cleaned rows:")
display(clean_df.head())

DATA CLEANING SUMMARY
---------------------------------------------
Rows before cleaning: 10000
Rows after cleaning:  10000
Duplicate rows removed: 0

Missing values after cleaning:
UDI                        0
Product ID                 0
Type                       0
Air temperature [K]        0
Process temperature [K]    0
Rotational speed [rpm]     0
Torque [Nm]                0
Tool wear [min]            0
Machine failure            0
TWF                        0
HDF                        0
PWF                        0
OSF                        0
RNF                        0
dtype: int64

First five cleaned rows:


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


## Step 2C: Validate the cleaned dataset

In [3]:
required_columns = feature_columns + ["Machine failure"]
numeric_columns = feature_columns

validation_results = {
    "Missing required columns": len([col for col in required_columns if col not in clean_df.columns]),
    "Missing values remaining": int(clean_df[required_columns].isna().sum().sum()),
    "Air temperature outside 250-350K": int((~clean_df["Air temperature [K]"].between(250, 350)).sum()),
    "Process temperature outside 250-350K": int((~clean_df["Process temperature [K]"].between(250, 350)).sum()),
    "Negative values in physical measurements": int((clean_df[numeric_columns] < 0).sum().sum()),
    "Invalid Machine failure labels": int((~clean_df["Machine failure"].isin([0, 1])).sum()),
    "Duplicate rows remaining": int(clean_df.duplicated().sum()),
}

print("CLEANED DATA VALIDATION")
print("-" * 45)
for check, count in validation_results.items():
    print(f"{check}: {count}")

if sum(validation_results.values()) == 0:
    print("\nVALIDATION PASSED: Cleaned data is ready for splitting.")
else:
    print("\nVALIDATION NEEDS ATTENTION: fix checks with non-zero values.")

CLEANED DATA VALIDATION
---------------------------------------------
Missing required columns: 0
Missing values remaining: 0
Air temperature outside 250-350K: 0
Process temperature outside 250-350K: 0
Negative values in physical measurements: 0
Invalid Machine failure labels: 0
Duplicate rows remaining: 0

VALIDATION PASSED: Cleaned data is ready for splitting.


## Step 2D: Check data types and class balance

In [4]:
print("Data types:")
print(clean_df.dtypes)

print("\nMachine failure class distribution after cleaning:")
print(clean_df["Machine failure"].value_counts().sort_index())

Data types:
UDI                          int64
Product ID                     str
Type                           str
Air temperature [K]        float64
Process temperature [K]    float64
Rotational speed [rpm]       int64
Torque [Nm]                float64
Tool wear [min]              int64
Machine failure              int64
TWF                          int64
HDF                          int64
PWF                          int64
OSF                          int64
RNF                          int64
dtype: object

Machine failure class distribution after cleaning:
Machine failure
0    9661
1     339
Name: count, dtype: int64


## Step 2E: Reproducible stratified train, validation and test splits

In [5]:
SEED = 42
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
TEST_RATIO = 0.15

def create_stratified_splits(dataframe):
    train_parts = []
    val_parts = []
    test_parts = []
    
    for _, group in dataframe.groupby("Machine failure", sort=True):
        shuffled = group.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        total = len(shuffled)
        test_n = round(total * TEST_RATIO)
        val_n = round(total * VALIDATION_RATIO)
        test_parts.append(shuffled.iloc[:test_n])
        val_parts.append(shuffled.iloc[test_n : test_n + val_n])
        train_parts.append(shuffled.iloc[test_n + val_n :])
        
    train_data = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    val_data = pd.concat(val_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    test_data = pd.concat(test_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return train_data, val_data, test_data

train_df, val_df, test_df = create_stratified_splits(clean_df)

assert len(train_df) + len(val_df) + len(test_df) == len(clean_df)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(PROCESSED_DIR / "predictive_maintenance_clean.csv", index=False)
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR / "validation.csv", index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv", index=False)

print("DATA SPLIT AND SAVE SUMMARY")
print("-" * 45)
print(f"Cleaned dataset:    {len(clean_df)} rows")
print(f"Training dataset:   {len(train_df)} rows")
print(f"Validation dataset: {len(val_df)} rows")
print(f"Test dataset:       {len(test_df)} rows")
print(f"Random seed used:   {SEED}")

print("\nClass distribution in each split:")
for split_name, split_data in [("Training", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"\n{split_name}:")
    print(split_data["Machine failure"].value_counts().sort_index())

DATA SPLIT AND SAVE SUMMARY
---------------------------------------------
Cleaned dataset:    10000 rows
Training dataset:   7000 rows
Validation dataset: 1500 rows
Test dataset:       1500 rows
Random seed used:   42

Class distribution in each split:

Training:
Machine failure
0    6763
1     237
Name: count, dtype: int64

Validation:
Machine failure
0    1449
1      51
Name: count, dtype: int64

Test:
Machine failure
0    1449
1      51
Name: count, dtype: int64


## Generated Testing Scenarios

The cited AI4I 2020 dataset contains 10,000 real records.

For stress-testing and simulation demonstration, 256 separate test scenarios are generated with controlled 1% IQR jitter. These scenarios are strictly for evaluation and demonstration and are NEVER mixed into training, validation, or test data.

In [6]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from generate_test_scenarios import run_generator

run_generator()

scenarios = pd.read_csv(PROJECT_ROOT / "data" / "generated" / "generated_predictive_maintenance_test_scenarios.csv")
print("Scenario table shape:", scenarios.shape)
print("\nExpected class balance:")
print(scenarios["Expected_Machine_Failure"].value_counts().sort_index())
print("\nFirst five generated testing scenarios:")
display(scenarios.head())

=== SC08 GENERATED TESTING SCENARIOS COMPLETED ===
Real cleaned records: 10000
Generated testing scenarios: 256
Generation seed: 42
Output file: C:\Users\DEEPAK\OneDrive\New folder\SC08-Predictive-Maintenance-System\data\generated\generated_predictive_maintenance_test_scenarios.csv
Scenario table shape: (256, 15)


Scenario table shape: (256, 15)

Expected class balance:
Expected_Machine_Failure
0    247
1      9
Name: count, dtype: int64

First five generated testing scenarios:


,Scenario_ID,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Expected_Machine_Failure,TWF,HDF,PWF,OSF,RNF
0,GEN-PM-001,7271,L54450,L,300.215895,310.429139,1808,25.714732,37,0,0,0,0,0,0
1,GEN-PM-002,861,H30274,H,296.095576,306.883724,1539,32.846493,33,0,0,0,0,0,0
2,GEN-PM-003,5391,H34804,H,302.820726,312.310208,1482,47.296254,223,0,0,0,0,0,0
3,GEN-PM-004,5192,L52371,L,304.048737,313.217817,1417,45.923505,128,0,0,0,0,0,0
4,GEN-PM-005,5735,L52914,L,302.292507,311.778681,1370,56.286007,209,1,0,0,0,1,0
